In [ ]:
import torch

model_name = 'meta-llama/Llama-2-7b-chat-hf'
# model_name = 'meta-llama/Llama-3.1-8B-Instruct'
# model_name = 'Qwen/Qwen3-4B'
# model_name = 'Qwen/Qwen3-1.7B'
clean = model_name.replace("/", "_").replace(":", "_").replace("@", "_")
chat = f'artifacts/residuals/{clean}/chat_completion_residuals.pt'
gen = f'artifacts/residuals/{clean}/completion_residuals.pt'
resid_chat = torch.load(chat)
resid_gen = torch.load(gen)

In [ ]:
resid_gen['pre'][0].shape

In [ ]:
chat_pre = []
gen_pre = []
for i in range(len(resid_chat['pre'])):
    chat_pre_i = resid_chat['pre'][i][:, -1]
    chat_pre.append(chat_pre_i)

    gen_pre_i = resid_gen['pre'][i][:, -1]
    gen_pre.append(gen_pre_i)

chat_pre = torch.stack(chat_pre)
gen_pre = torch.stack(gen_pre)

In [ ]:
chat_pre.shape, gen_pre.shape

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

print(f"Shape of chat_pre: {chat_pre.shape}")
print(f"Shape of gen_pre: {gen_pre.shape}")

# Combine both sets of trajectories into a single tensor for PCA
# Shape will be (200, 32, 4096)
all_trajectories_tensor = torch.cat([chat_pre, gen_pre], dim=0)

num_trajectories, num_layers, hidden_dim = all_trajectories_tensor.shape
reshaped_for_pca = all_trajectories_tensor.reshape(-1, hidden_dim)

reshaped_for_pca_np = reshaped_for_pca


print("\nFitting PCA on all 6400 trajectory points...")
pca = PCA(n_components=2)
projected_trajectories = pca.fit_transform(reshaped_for_pca_np)

projected_trajectories = projected_trajectories.reshape(num_trajectories, num_layers, -1)
print("PCA fitting complete.")

In [ ]:
projected_trajectories.shape

In [ ]:
comp = resid_gen['pre'][0][:, -1]
ref = resid_gen['pre'][0][:, 29]

vec = comp - ref
print(vec.shape)
new = pca.transform(vec)

comp_pca = pca.transform(comp)
ref_pca = pca.transform(ref)

In [ ]:
# Plot the trajectories
plt.figure(figsize=(12, 8))

# Split the projected trajectories back into chat and gen
chat_projected = projected_trajectories[:100]  # First 100 are chat
gen_projected = projected_trajectories[100:]   # Next 100 are gen

# Filter out trajectories where any point has PC1 > -125
chat_mask = (chat_projected[:, :, 0] <= -125).all(axis=1)
gen_mask = (gen_projected[:, :, 0] <= -125).all(axis=1)

chat_projected = chat_projected[chat_mask]
gen_projected = gen_projected[gen_mask]

# Plot chat trajectories
for i in range(len(chat_projected)):
    label = 'Setup A' if i == 0 else None
    alpha = 0.8 if i == 0 else 0.3
    plt.plot(chat_projected[i, :, 0], chat_projected[i, :, 1], 'r-', alpha=alpha, linewidth=0.5, label=label)
    # Mark start and end points
    if i == 0:  # Only label the first trajectory for clarity
        plt.plot(chat_projected[i, 0, 0], chat_projected[i, 0, 1], 'go', alpha=alpha, markersize=3, label='Start (o)')
        plt.plot(chat_projected[i, -1, 0], chat_projected[i, -1, 1], 'gx', alpha=alpha, markersize=3, label='End (x)')
    else:
        plt.plot(chat_projected[i, 0, 0], chat_projected[i, 0, 1], 'ro', alpha=alpha, markersize=3)
        plt.plot(chat_projected[i, -1, 0], chat_projected[i, -1, 1], 'rx', alpha=alpha, markersize=3)

# Plot gen trajectories  
gen_label_added = False
for i in range(len(gen_projected)):
    if i == 50:
        continue
    label = 'Setup B' if not gen_label_added else None
    alpha = 0.8 if not gen_label_added else 0.3
    if not gen_label_added:
        gen_label_added = True
    plt.plot(gen_projected[i, :, 0], gen_projected[i, :, 1], 'b-', alpha=alpha, linewidth=0.5, label=label)
    # Mark start and end points
    plt.plot(gen_projected[i, 0, 0], gen_projected[i, 0, 1], 'bo', alpha=alpha, markersize=3)
    plt.plot(gen_projected[i, -1, 0], gen_projected[i, -1, 1], 'bx', alpha=alpha, markersize=3)


# plt.plot(new[:, 0], new[:, 1], 'go-', alpha=0.3, linewidth=1.5, label='diff')
# plt.plot(comp_pca[:, 0], comp_pca[:, 1], 'o-', color='yellow', alpha=0.7, linewidth=1.5, label='Comp')
# plt.plot(ref_pca[:, 0], ref_pca[:, 1], 'o-', color='purple', alpha=0.7, linewidth=1.5, label='Ref')
# plt.plot(vec_pca[:, 0], vec_pca[:, 1], color='black', alpha=0.7, linewidth=1.5, label='Vec')

# plt.xlim(-150, -125)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Trajectory Visualization (Harmless Prompts)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(f'artifacts/figures/traj_harmless.svg')
plt.show()

In [ ]:
# Plot the trajectories
plt.figure(figsize=(12, 8))

# Split the projected trajectories back into chat and gen
chat_projected = projected_trajectories[:100]  # First 100 are chat
gen_projected = projected_trajectories[100:]   # Next 100 are gen

# Plot chat trajectories
for i in range(len(chat_projected)):
    label = 'Setup A' if i == 0 else None
    alpha = 0.8 if i == 0 else 0.3
    plt.plot(chat_projected[i, :, 0], chat_projected[i, :, 1], 'r-', alpha=alpha, linewidth=0.5, label=label)
    # Mark start and end points
    if i == 0:  # Only label the first trajectory for clarity
        plt.plot(chat_projected[i, 0, 0], chat_projected[i, 0, 1], 'go', alpha=alpha, markersize=3, label='Start (o)')
        plt.plot(chat_projected[i, -1, 0], chat_projected[i, -1, 1], 'gx', alpha=alpha, markersize=3, label='End (x)')
    else:
        plt.plot(chat_projected[i, 0, 0], chat_projected[i, 0, 1], 'ro', alpha=alpha, markersize=3)
        plt.plot(chat_projected[i, -1, 0], chat_projected[i, -1, 1], 'rx', alpha=alpha, markersize=3)

# Plot gen trajectories  
gen_label_added = False
for i in range(len(gen_projected)):
    if i == 50:
        continue
    label = 'Setup B' if not gen_label_added else None
    alpha = 0.8 if not gen_label_added else 0.3
    if not gen_label_added:
        gen_label_added = True
    plt.plot(gen_projected[i, :, 0], gen_projected[i, :, 1], 'b-', alpha=alpha, linewidth=0.5, label=label)
    # Mark start and end points
    plt.plot(gen_projected[i, 0, 0], gen_projected[i, 0, 1], 'bo', alpha=alpha, markersize=3)
    plt.plot(gen_projected[i, -1, 0], gen_projected[i, -1, 1], 'bx', alpha=alpha, markersize=3)


# plt.plot(new[:, 0], new[:, 1], 'go-', alpha=0.3, linewidth=1.5, label='diff')
# plt.plot(comp_pca[:, 0], comp_pca[:, 1], 'o-', color='yellow', alpha=0.7, linewidth=1.5, label='Comp')
# plt.plot(ref_pca[:, 0], ref_pca[:, 1], 'o-', color='purple', alpha=0.7, linewidth=1.5, label='Ref')
# plt.plot(vec_pca[:, 0], vec_pca[:, 1], color='black', alpha=0.7, linewidth=1.5, label='Vec')


plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Trajectory Visualization (Harmful Prompts)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('artifacts/figures/traj_harmful.svg')
plt.show()

In [ ]:
projected_trajectories.shape, all_trajectories_tensor.shape

In [ ]:
chat_pre.shape

In [ ]:
# Calculate cosine similarity between chat_mean and gen_mean for each layer
# Calculate mean trajectories for each class

# Find indices to skip based on projected trajectories filtering
chat_projected_full = projected_trajectories[:100]  # First 100 are chat
gen_projected_full = projected_trajectories[100:]   # Next 100 are gen

# Find which trajectories to keep (where all PC1 values <= -125)
chat_keep_mask = (chat_projected_full[:, :, 0] <= -125).all(axis=1)
gen_keep_mask = (gen_projected_full[:, :, 0] <= -125).all(axis=1)

# Apply the same filtering to the pre-projection data
chat_filtered = chat_pre[chat_keep_mask]
gen_filtered = gen_pre[gen_keep_mask]

chat_mean = torch.mean(chat_filtered, dim=0)  # Shape: (32, 4096)
gen_mean = torch.mean(gen_filtered, dim=0)    # Shape: (32, 4096)

cosine_similarities = []
for layer in range(32):
    # Get the vectors for this layer
    chat_vec = chat_mean[layer]  # Shape: (4096,)
    gen_vec = gen_mean[layer]    # Shape: (4096,)
    
    # Calculate cosine similarity
    cosine_sim = torch.nn.functional.cosine_similarity(chat_vec.unsqueeze(0), gen_vec.unsqueeze(0), dim=1)
    cosine_similarities.append(cosine_sim.item())

# Convert to numpy for plotting
cosine_similarities = np.array(cosine_similarities)

# Plot cosine similarity vs layers
plt.figure(figsize=(10, 6))
plt.plot(range(32), cosine_similarities, 'o-', linewidth=2, markersize=6)
plt.xlabel('Layer')
plt.ylabel('Cosine Similarity')
plt.title('Cosine Similarity (Harmless Prompts)')
plt.grid(True, alpha=0.3)
plt.xlim(-0.5, 31.5)
plt.savefig('artifacts/figures/cosine_similarity_harmless.svg')
plt.show()

print(f"Cosine similarities shape: {cosine_similarities.shape}")
print(f"Min cosine similarity: {cosine_similarities.min():.4f}")
print(f"Max cosine similarity: {cosine_similarities.max():.4f}")
print(f"Mean cosine similarity: {cosine_similarities.mean():.4f}")


In [ ]:
# Calculate cosine similarity between chat_mean and gen_mean for each layer
# Calculate mean trajectories for each class
chat_mean = torch.mean(chat_pre, dim=0)  # Shape: (32, 4096)

# Filter out trajectory 50 from gen_pre before calculating mean
gen_filtered = torch.cat([gen_pre[:50], gen_pre[51:]], dim=0)  # Remove trajectory at index 50
gen_mean = torch.mean(gen_pre, dim=0)    # Shape: (32, 4096)

cosine_similarities = []
for layer in range(32):
    # Get the vectors for this layer
    chat_vec = chat_mean[layer]  # Shape: (4096,)
    gen_vec = gen_mean[layer]    # Shape: (4096,)
    
    # Calculate cosine similarity
    cosine_sim = torch.nn.functional.cosine_similarity(chat_vec.unsqueeze(0), gen_vec.unsqueeze(0), dim=1)
    cosine_similarities.append(cosine_sim.item())

# Convert to numpy for plotting
cosine_similarities = np.array(cosine_similarities)

# Plot cosine similarity vs layers
plt.figure(figsize=(10, 6))
plt.plot(range(32), cosine_similarities, 'o-', linewidth=2, markersize=6)
plt.xlabel('Layer')
plt.ylabel('Cosine Similarity')
plt.title('Cosine Similarity (Harm Prompts)')
plt.grid(True, alpha=0.3)
plt.xlim(-0.5, 31.5)
plt.savefig('artifacts/figures/cosine_similarity_harmful.svg')
plt.show()

print(f"Cosine similarities shape: {cosine_similarities.shape}")
print(f"Min cosine similarity: {cosine_similarities.min():.4f}")
print(f"Max cosine similarity: {cosine_similarities.max():.4f}")
print(f"Mean cosine similarity: {cosine_similarities.mean():.4f}")
